# 03. Preparación de datos para el modelado — IT

Este notebook recibe como entrada `Tabla_Entrenamiento_IT.csv` y genera como salida una **tabla de entrenamiento preparada para la fase de modelado**.

## Flujo
1. Carga de `Tabla_Entrenamiento_IT.csv`.
2. Retirada inicial de variables que no participarán en la predicción.
3. Separación temporal de las cuatro variables objetivo.
4. Análisis exploratorio inicial de las variables candidatas.
5. Tratamiento de valores faltantes.
6. Tratamiento de anomalías de `Edad`.
7. Reducción de dimensionalidad de categorías poco frecuentes mediante umbrales diferenciados.
8. Análisis de las variables categóricas después de la reducción.
9. Reincorporación de las cuatro variables objetivo.
10. Validaciones y exportación de la tabla de entrenamiento final.

**Importante:** no se realiza One-Hot Encoding ni escalado en esta etapa. Estas transformaciones se realizarán en el pipeline de modelado, ajustándolas únicamente con los datos de entrenamiento para evitar fuga de información.


In [2]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

INPUT_FILE = Path("Tabla_Entrenamiento_IT.csv")
OUTPUT_DIR = Path("Resultados_03_Preparacion_Datos_IT")
TABLE_DIR = OUTPUT_DIR / "tablas"
GRAPH_DIR = OUTPUT_DIR / "graficos"

for folder in [OUTPUT_DIR, TABLE_DIR, GRAPH_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"No se encontró {INPUT_FILE.resolve()}. "
        "Coloca el notebook en la misma carpeta que Tabla_Entrenamiento_IT.csv."
    )

df = pd.read_csv(INPUT_FILE, sep=";", low_memory=False)

print(f"Archivo: {INPUT_FILE}")
print(f"Filas: {len(df):,}")
print(f"Columnas: {len(df.columns)}")


Archivo: Tabla_Entrenamiento_IT.csv
Filas: 1,755,108
Columnas: 56


## 1. Definición de variables

Las variables se separan en tres grupos:

- **Variables excluidas:** identificadores, campos administrativos/técnicos, variables de control y `UltimoStatus`.
- **Variables candidatas:** 36 variables que quedan disponibles para el modelado.
- **Variables objetivo:** `IT_Futura_W1` a `IT_Futura_W4`.

Las variables objetivo se separan temporalmente durante el tratamiento, pero se reincorporan al output final.


In [3]:
TARGETS = [
    "IT_Futura_W1",
    "IT_Futura_W2",
    "IT_Futura_W3",
    "IT_Futura_W4",
]

EXCLUDED = [
    "CodEmpleado",
    "IdAlta",
    "IdContrato",
    "FechaInicioAlta",
    "FechaFinAlta",
    "ParticipacionDesde",
    "ParticipacionHasta",
    "FechaInicio",
    "FechaFin",
    "OrganizacionDesdeAsignada",
    "OrganizacionHastaAsignada",
    "OrigenAsignacionOrganizativa",
    "Nacimiento",
    "ITAbierto",
    "ElegiblePrediccion",
    "UltimoStatus",
]

CANDIDATES = [
    "AnioISO",
    "SemanaISO",
    "ID_Semana",
    "ClaseContrato",
    "CantidadPeriodosContrato",
    "DiasTrabajadosHastaCorte",
    "FamiliaFuncion",
    "Centro",
    "UnidadOrganizativa",
    "Funcion",
    "Area",
    "Servicio",
    "Sexo",
    "Edad",
    "TipoJornada",
    "PorcHorTbjo",
    "HSem",
    "HTbjoAnio",
    "Previous1WeekIT",
    "Previous1WeekITShortTerm",
    "Previous1WeekITLongTerm",
    "Previous2WeeksIT",
    "Previous2WeeksITShortTerm",
    "Previous2WeeksITLongTerm",
    "Previous1MonthIT",
    "Previous1MonthITShortTerm",
    "Previous1MonthITLongTerm",
    "Previous3MonthsIT",
    "Previous3MonthsITShortTerm",
    "Previous3MonthsITLongTerm",
    "Previous6MonthsIT",
    "Previous6MonthsITShortTerm",
    "Previous6MonthsITLongTerm",
    "Previous12MonthsIT",
    "Previous12MonthsITShortTerm",
    "Previous12MonthsITLongTerm",
]

expected = set(TARGETS + EXCLUDED + CANDIDATES)
missing = sorted(expected - set(df.columns))

if missing:
    raise ValueError(f"Faltan columnas esperadas en el CSV: {missing}")

print("✓ Las columnas esperadas están presentes.")
print(f"Variables excluidas: {len(EXCLUDED)}")
print(f"Variables candidatas: {len(CANDIDATES)}")
print(f"Variables objetivo: {len(TARGETS)}")


✓ Las columnas esperadas están presentes.
Variables excluidas: 16
Variables candidatas: 36
Variables objetivo: 4


In [4]:
# Tabla de control de estructura
estructura = pd.DataFrame({
    "Grupo": ["Excluidas", "Candidatas", "Objetivo"],
    "N_variables": [len(EXCLUDED), len(CANDIDATES), len(TARGETS)]
})
display(estructura)

# Se trabaja exclusivamente con las candidatas y las targets.
X = df[CANDIDATES].copy()
y = df[TARGETS].copy()

print(f"X: {X.shape}")
print(f"y: {y.shape}")


,Grupo,N_variables
0,Excluidas,16
1,Candidatas,36
2,Objetivo,4


X: (1755108, 36)
y: (1755108, 4)


## 2. Depuración inicial

Se eliminan del conjunto de trabajo las variables que no participarán como predictores. Las cuatro variables objetivo se mantienen separadas para evitar tratarlas como variables explicativas.

Resultado esperado en esta fase:

**36 variables candidatas + 4 variables objetivo = 40 columnas relevantes para modelado.**


In [5]:
# Comprobación explícita de que ninguna variable excluida queda en X
assert not (set(EXCLUDED) & set(X.columns)), "Hay variables excluidas dentro de X."

# Comprobación de que las targets no forman parte de X
assert not (set(TARGETS) & set(X.columns)), "Hay targets dentro de X."

print("✓ Depuración inicial validada.")


✓ Depuración inicial validada.


## 3. Análisis exploratorio inicial

Este análisis tiene carácter descriptivo. No se realiza selección definitiva de predictores.

Se revisan:
- tipos de datos;
- valores faltantes;
- cardinalidad de variables categóricas;
- estadísticos de variables numéricas;
- distribución de las variables históricas de IT.


In [6]:
# Tipos de datos
tipos = pd.DataFrame({
    "Variable": X.columns,
    "Tipo": X.dtypes.astype(str).values,
    "Nulos": X.isna().sum().values,
    "Porcentaje_Nulos": (X.isna().mean().values * 100)
}).sort_values(["Tipo", "Variable"])

display(tipos)
tipos.to_csv(TABLE_DIR / "01_tipos_y_nulos_inicial.csv", sep=";", index=False, encoding="utf-8-sig")


,Variable,Tipo,Nulos,Porcentaje_Nulos
13,Edad,float64,27,0.001538
16,HSem,float64,27,0.001538
17,HTbjoAnio,float64,27,0.001538
15,PorcHorTbjo,float64,27,0.001538
0,AnioISO,int64,0,0.000000
4,CantidadPeriodosContrato,int64,0,0.000000
5,DiasTrabajadosHastaCorte,int64,0,0.000000
2,ID_Semana,int64,0,0.000000
33,Previous12MonthsIT,int64,0,0.000000
35,Previous12MonthsITLongTerm,int64,0,0.000000


In [7]:
NUMERICAS = [
    "AnioISO", "SemanaISO", "ID_Semana",
    "CantidadPeriodosContrato", "DiasTrabajadosHastaCorte",
    "Edad", "PorcHorTbjo", "HSem", "HTbjoAnio",
    "Previous1WeekIT", "Previous1WeekITShortTerm", "Previous1WeekITLongTerm",
    "Previous2WeeksIT", "Previous2WeeksITShortTerm", "Previous2WeeksITLongTerm",
    "Previous1MonthIT", "Previous1MonthITShortTerm", "Previous1MonthITLongTerm",
    "Previous3MonthsIT", "Previous3MonthsITShortTerm", "Previous3MonthsITLongTerm",
    "Previous6MonthsIT", "Previous6MonthsITShortTerm", "Previous6MonthsITLongTerm",
    "Previous12MonthsIT", "Previous12MonthsITShortTerm", "Previous12MonthsITLongTerm",
]

CATEGORICAS = [
    "ClaseContrato", "FamiliaFuncion", "Centro", "UnidadOrganizativa",
    "Funcion", "Area", "Servicio", "Sexo", "TipoJornada"
]

assert set(NUMERICAS) | set(CATEGORICAS) == set(CANDIDATES)
assert not (set(NUMERICAS) & set(CATEGORICAS))

print(f"Numéricas: {len(NUMERICAS)}")
print(f"Categóricas: {len(CATEGORICAS)}")


Numéricas: 27
Categóricas: 9


In [8]:
# Estadísticos numéricos
numeric_stats = X[NUMERICAS].describe().T
numeric_stats["Nulos"] = X[NUMERICAS].isna().sum()
numeric_stats["Porcentaje_Nulos"] = X[NUMERICAS].isna().mean() * 100
display(numeric_stats)
numeric_stats.to_csv(TABLE_DIR / "02_estadisticos_numericos_inicial.csv", sep=";", encoding="utf-8-sig")


,count,mean,std,min,25%,50%,75%,max,Nulos,Porcentaje_Nulos
AnioISO,1755108.0,2024.824562,0.753458,2024.00,2024.00,2025.0,2025.0,2026.0,0,0.000000
SemanaISO,1755108.0,24.070469,14.695350,1.00,12.00,23.0,36.0,52.0,0,0.000000
ID_Semana,1755108.0,66.947712,37.724481,1.00,34.00,68.0,100.0,131.0,0,0.000000
CantidadPeriodosContrato,1755108.0,3.604883,4.130576,1.00,1.00,3.0,5.0,288.0,0,0.000000
DiasTrabajadosHastaCorte,1755108.0,2566.296166,2698.045933,1.00,612.00,1778.0,3468.0,18541.0,0,0.000000
Edad,1755081.0,39.003769,11.433715,0.00,30.00,37.0,47.0,99.0,27,0.001538
PorcHorTbjo,1755081.0,92.455062,18.931526,5.00,100.00,100.0,100.0,100.0,27,0.001538
HSem,1755081.0,35.565091,7.452240,1.88,37.25,37.5,40.0,40.0,27,0.001538
HTbjoAnio,1755081.0,1543.144592,319.241425,79.00,1642.50,1680.0,1680.0,1816.0,27,0.001538
Previous1WeekIT,1755108.0,0.282864,1.130642,0.00,0.00,0.0,0.0,5.0,0,0.000000


In [9]:
# Cardinalidad y frecuencia de categóricas
cat_rows = []

for col in CATEGORICAS:
    vc = X[col].value_counts(dropna=False)
    non_null = X[col].dropna()
    counts = non_null.value_counts()

    cat_rows.append({
        "Variable": col,
        "N_categorias": non_null.nunique(),
        "Categorias_menos_10": int((counts < 10).sum()),
        "Categorias_menos_50": int((counts < 50).sum()),
        "Categorias_menos_100": int((counts < 100).sum()),
        "Categoria_mas_frecuente": str(counts.index[0]) if len(counts) else "",
        "Frecuencia_mas_frecuente": int(counts.iloc[0]) if len(counts) else 0,
        "Porcentaje_mas_frecuente": float(counts.iloc[0] / len(non_null) * 100) if len(counts) else np.nan,
        "Nulos": int(X[col].isna().sum()),
    })

cat_cardinality = pd.DataFrame(cat_rows)
display(cat_cardinality)
cat_cardinality.to_csv(TABLE_DIR / "03_cardinalidad_categoricas_inicial.csv", sep=";", index=False, encoding="utf-8-sig")


,Variable,N_categorias,Categorias_menos_10,Categorias_menos_50,Categorias_menos_100,Categoria_mas_frecuente,Frecuencia_mas_frecuente,Porcentaje_mas_frecuente,Nulos
0,ClaseContrato,2,0,0,0,ilimitado,1542060,87.861260,0
1,FamiliaFuncion,17,0,0,0,ENFERMERÍA,401223,22.861735,110
2,Centro,21,0,0,0,Centro 14,579908,33.041158,0
3,UnidadOrganizativa,594,16,38,58,URGENCIAS,98700,5.623970,120
4,Funcion,452,7,24,50,ENFERMERO/A HOSPITALIZACIÓN,146865,8.368386,110
5,Area,80,0,4,5,DIRECCIÓN MÉDICA,353282,20.128790,0
6,Servicio,200,1,7,12,URGENCIAS,108658,6.190958,0
7,Sexo,2,0,0,0,femenino,1310621,74.675813,27
8,TipoJornada,17,1,2,2,L-V 08:00-15:00,955114,54.419938,27


## 4. Tratamiento de valores faltantes

Se conserva el conjunto de registros y se imputan los pocos valores faltantes:

- Variables numéricas → **media**.
- Variables categóricas → **moda**.

La imputación se realiza sobre las variables candidatas, nunca sobre las variables objetivo.


In [10]:
# Nulos antes del tratamiento
nulos_antes = X.isna().sum()
nulos_antes = nulos_antes[nulos_antes > 0].sort_values(ascending=False)

if len(nulos_antes):
    display(pd.DataFrame({
        "Nulos": nulos_antes,
        "Porcentaje": (nulos_antes / len(X) * 100).round(6)
    }))
else:
    print("No hay valores faltantes en las variables candidatas.")


,Nulos,Porcentaje
UnidadOrganizativa,120,0.006837
FamiliaFuncion,110,0.006267
Funcion,110,0.006267
Sexo,27,0.001538
Edad,27,0.001538
TipoJornada,27,0.001538
PorcHorTbjo,27,0.001538
HSem,27,0.001538
HTbjoAnio,27,0.001538


In [11]:
imputation_log = []

# Numéricas -> media
for col in NUMERICAS:
    n = int(X[col].isna().sum())
    if n > 0:
        mean_value = X[col].mean()
        X[col] = X[col].fillna(mean_value)
        imputation_log.append({
            "Variable": col,
            "Tipo": "Numerica",
            "Metodo": "Media",
            "N_imputados": n,
            "Valor_imputacion": float(mean_value)
        })

# Categóricas -> moda
for col in CATEGORICAS:
    n = int(X[col].isna().sum())
    if n > 0:
        mode_series = X[col].mode(dropna=True)
        if mode_series.empty:
            raise ValueError(f"No se pudo obtener la moda para {col}.")
        mode_value = mode_series.iloc[0]
        X[col] = X[col].fillna(mode_value)
        imputation_log.append({
            "Variable": col,
            "Tipo": "Categorica",
            "Metodo": "Moda",
            "N_imputados": n,
            "Valor_imputacion": str(mode_value)
        })

imputation_log_df = pd.DataFrame(imputation_log)
display(imputation_log_df if len(imputation_log_df) else pd.DataFrame(
    columns=["Variable","Tipo","Metodo","N_imputados","Valor_imputacion"]
))

imputation_log_df.to_csv(TABLE_DIR / "04_log_imputacion.csv", sep=";", index=False, encoding="utf-8-sig")

assert X.isna().sum().sum() == 0
print("✓ No quedan valores faltantes en las 36 variables candidatas.")


,Variable,Tipo,Metodo,N_imputados,Valor_imputacion
0,Edad,Numerica,Media,27,39.003769
1,PorcHorTbjo,Numerica,Media,27,92.455062
2,HSem,Numerica,Media,27,35.565091
3,HTbjoAnio,Numerica,Media,27,1543.144592
4,FamiliaFuncion,Categorica,Moda,110,ENFERMERÍA
5,UnidadOrganizativa,Categorica,Moda,120,URGENCIAS
6,Funcion,Categorica,Moda,110,ENFERMERO/A HOSPITALIZACIÓN
7,Sexo,Categorica,Moda,27,femenino
8,TipoJornada,Categorica,Moda,27,L-V 08:00-15:00


✓ No quedan valores faltantes en las 36 variables candidatas.


## 5. Tratamiento de anomalías en `Edad`

Se mantiene el criterio definido para la preparación:

- `Edad` igual a 0 o 1 → valor inconsistente; se sustituye por la media de edad válida.
- `Edad` superior a 80 → se limita a 80.

No se aplican transformaciones automáticas a otras variables únicamente por presentar valores extremos; cualquier tratamiento adicional deberá justificarse en función del contexto y/o validarse durante el modelado.


In [12]:
edad = X["Edad"].copy()

edad_valid = edad[(edad > 1) & (edad <= 80)]
if edad_valid.empty:
    raise ValueError("No hay valores válidos de Edad para calcular la media.")

edad_mean = float(edad_valid.mean())

n_01 = int(((edad == 0) | (edad == 1)).sum())
n_gt80 = int((edad > 80).sum())

X.loc[(X["Edad"] == 0) | (X["Edad"] == 1), "Edad"] = edad_mean
X.loc[X["Edad"] > 80, "Edad"] = 80

edad_log = pd.DataFrame([{
    "Tratamiento": "Edad 0/1 -> media",
    "N_registros": n_01,
    "Valor_aplicado": edad_mean
}, {
    "Tratamiento": "Edad >80 -> 80",
    "N_registros": n_gt80,
    "Valor_aplicado": 80
}])

display(edad_log)
edad_log.to_csv(TABLE_DIR / "05_tratamiento_edad.csv", sep=";", index=False, encoding="utf-8-sig")

print(f"Media de edad válida utilizada: {edad_mean:.6f}")
print(f"Registros 0/1 tratados: {n_01:,}")
print(f"Registros >80 tratados: {n_gt80:,}")


,Tratamiento,N_registros,Valor_aplicado
0,Edad 0/1 -> media,2,38.97921
1,Edad >80 -> 80,815,80.00000


Media de edad válida utilizada: 38.979210
Registros 0/1 tratados: 2
Registros >80 tratados: 815


## 6. Reducción de dimensionalidad de las variables categóricas

Tras el análisis exploratorio se identificó una elevada cardinalidad en algunas variables organizativas. Para reducir la dimensionalidad antes de la codificación, se agrupan las categorías con una frecuencia inferior a un umbral determinado en una única categoría denominada `Otros`.

A partir del análisis de sensibilidad realizado sobre la cardinalidad, se establecen inicialmente **umbrales diferenciados según la variable**, buscando un equilibrio entre la reducción del número de categorías y la proporción de observaciones afectadas:

| Variable | Umbral de agrupación |
|---|---:|
| `TipoJornada` | <500 |
| `Area` | <1.000 |
| `Servicio` | <1.000 |
| `Funcion` | <500 |
| `UnidadOrganizativa` | <500 |

Las variables `ClaseContrato`, `FamiliaFuncion`, `Centro` y `Sexo` no se agrupan, al no obtenerse un beneficio relevante de esta transformación.

La agrupación modifica únicamente las categorías de las variables, **sin eliminar ninguna variable candidata**. La utilidad predictiva definitiva de estas variables se evaluará posteriormente durante el modelado.


In [13]:
# Umbrales diferenciados obtenidos a partir del análisis de sensibilidad
RARE_THRESHOLDS = {
    "TipoJornada": 500,
    "Area": 1000,
    "Servicio": 1000,
    "Funcion": 500,
    "UnidadOrganizativa": 500,
}

grouping_rows = []
rare_categories_by_var = {}

for col, umbral in RARE_THRESHOLDS.items():
    counts = X[col].value_counts(dropna=False)
    rare = counts[counts < umbral].index.tolist()
    rare_categories_by_var[col] = rare

    grouping_rows.append({
        "Variable": col,
        "Umbral": umbral,
        "Categorias_antes": int(len(counts)),
        "Categorias_agrupadas": int(len(rare)),
        "Observaciones_a_Otros": int(counts[counts < umbral].sum()),
        "Observaciones_a_Otros_pct": float(counts[counts < umbral].sum() / len(X) * 100),
    })

grouping_before = pd.DataFrame(grouping_rows)
display(grouping_before)


,Variable,Umbral,Categorias_antes,Categorias_agrupadas,Observaciones_a_Otros,Observaciones_a_Otros_pct
0,TipoJornada,500,17,6,1495,0.085180
1,Area,1000,80,33,11179,0.636941
2,Servicio,1000,200,79,31573,1.798921
3,Funcion,500,452,240,49305,2.809229
4,UnidadOrganizativa,500,594,245,47485,2.705532


In [14]:
# Aplicación de la reducción de dimensionalidad
for col, umbral in RARE_THRESHOLDS.items():
    rare = set(rare_categories_by_var[col])
    X[col] = X[col].where(~X[col].isin(rare), "Otros")

# Resumen posterior
grouping_after_rows = []

for col, umbral in RARE_THRESHOLDS.items():
    counts_after = X[col].value_counts(dropna=False)
    categorias_antes = int(
        grouping_before.loc[
            grouping_before["Variable"] == col, "Categorias_antes"
        ].iloc[0]
    )

    grouping_after_rows.append({
        "Variable": col,
        "Umbral": umbral,
        "Categorias_antes": categorias_antes,
        "Categorias_despues": int(X[col].nunique(dropna=False)),
        "Reduccion_categorias": int(
            categorias_antes - X[col].nunique(dropna=False)
        ),
        "Reduccion_categorias_pct": float(
            (categorias_antes - X[col].nunique(dropna=False))
            / categorias_antes * 100
        ),
        "Observaciones_a_Otros": int((X[col] == "Otros").sum()),
        "Observaciones_a_Otros_pct": float((X[col] == "Otros").mean() * 100),
    })

grouping_summary = pd.DataFrame(grouping_after_rows)
display(grouping_summary)

grouping_summary.to_csv(
    TABLE_DIR / "06_reduccion_dimensionalidad_categoricas.csv",
    sep=";", index=False, encoding="utf-8-sig"
)


,Variable,Umbral,Categorias_antes,Categorias_despues,Reduccion_categorias,Reduccion_categorias_pct,Observaciones_a_Otros,Observaciones_a_Otros_pct
0,TipoJornada,500,17,12,5,29.411765,1495,0.085180
1,Area,1000,80,48,32,40.000000,11179,0.636941
2,Servicio,1000,200,122,78,39.000000,31573,1.798921
3,Funcion,500,452,213,239,52.876106,49305,2.809229
4,UnidadOrganizativa,500,594,350,244,41.077441,47485,2.705532


In [15]:
# Validación de la agrupación
for col, rare in rare_categories_by_var.items():
    remaining = set(rare).intersection(set(X[col].unique()))
    assert remaining <= {"Otros"}, (
        f"En {col} quedaron categorías raras sin agrupar: {remaining}"
    )

assert X[CANDIDATES].isna().sum().sum() == 0

print("✓ Reducción de dimensionalidad aplicada correctamente.")


✓ Reducción de dimensionalidad aplicada correctamente.


## 7. Análisis de las variables categóricas después del ajuste

Una vez aplicada la reducción de dimensionalidad, se revisa nuevamente la cardinalidad de las variables categóricas para comprobar el efecto de la transformación.

Este análisis permite documentar el número final de categorías que entrarán en la fase de codificación y verificar que la agrupación se haya aplicado únicamente a las variables definidas.


In [16]:
# Cardinalidad final de todas las variables categóricas
cardinality_after = []

for col in CATEGORICAS:
    cardinality_after.append({
        "Variable": col,
        "Categorias_finales": int(X[col].nunique(dropna=False)),
        "Categoria_mas_frecuente": str(X[col].value_counts(dropna=False).index[0]),
        "Frecuencia_mas_frecuente": int(X[col].value_counts(dropna=False).iloc[0]),
        "Porcentaje_mas_frecuente": float(
            X[col].value_counts(dropna=False).iloc[0] / len(X) * 100
        ),
    })

cardinality_final = pd.DataFrame(cardinality_after)
display(cardinality_final)

cardinality_final.to_csv(
    TABLE_DIR / "07_analisis_categoricas_ajustadas.csv",
    sep=";", index=False, encoding="utf-8-sig"
)


,Variable,Categorias_finales,Categoria_mas_frecuente,Frecuencia_mas_frecuente,Porcentaje_mas_frecuente
0,ClaseContrato,2,ilimitado,1542060,87.861260
1,FamiliaFuncion,17,ENFERMERÍA,401333,22.866570
2,Centro,21,Centro 14,579908,33.041158
3,UnidadOrganizativa,350,URGENCIAS,98820,5.630423
4,Funcion,213,ENFERMERO/A HOSPITALIZACIÓN,146975,8.374129
5,Area,48,DIRECCIÓN MÉDICA,353282,20.128790
6,Servicio,122,URGENCIAS,108658,6.190958
7,Sexo,2,femenino,1310648,74.676202
8,TipoJornada,12,L-V 08:00-15:00,955141,54.420640


## 8. Reincorporación de las variables objetivo

Las cuatro variables objetivo se reincorporan al conjunto final después de completar el tratamiento de las variables candidatas.

Las variables objetivo se mantienen exactamente como fueron generadas en la etapa de construcción de la tabla de entrenamiento y no participan en los tratamientos aplicados a los predictores.


In [17]:
# Mantener las targets exactamente como fueron recibidas
for col in TARGETS:
    assert y[col].equals(df[col]), f"La target {col} ha sido modificada accidentalmente."

df_model = pd.concat([X, y], axis=1)

assert df_model.shape[1] == 40
assert df_model.shape[0] == df.shape[0]
assert list(df_model.columns) == CANDIDATES + TARGETS

print(f"✓ Tabla final: {df_model.shape[0]:,} filas x {df_model.shape[1]} columnas")


✓ Tabla final: 1,755,108 filas x 40 columnas


## 9. Validaciones finales

Se comprueba que la tabla final:

- conserva todas las observaciones;
- contiene las 36 variables candidatas preparadas;
- contiene las cuatro variables objetivo;
- no contiene las variables excluidas;
- no presenta valores faltantes en los predictores;
- mantiene las variables objetivo sin modificaciones.


In [18]:
# Estructura
assert set(CANDIDATES).issubset(df_model.columns)
assert set(TARGETS).issubset(df_model.columns)
assert not (set(EXCLUDED) & set(df_model.columns))

# Nulos en predictores
candidate_nulls = df_model[CANDIDATES].isna().sum().sum()
assert candidate_nulls == 0, f"Quedan {candidate_nulls} nulos en predictores."

# Filas
assert len(df_model) == len(df)

# Edad
assert df_model["Edad"].min() > 1
assert df_model["Edad"].max() <= 80

# Targets: no deben contener nulos en esta tabla de entrenamiento
target_nulls = df_model[TARGETS].isna().sum()
display(pd.DataFrame({"Nulos": target_nulls, "Porcentaje": target_nulls / len(df_model) * 100}))

if target_nulls.sum() > 0:
    print("AVISO: hay targets nulas. Revisar cobertura de las semanas futuras antes del modelado.")
else:
    print("✓ Las cuatro targets no contienen nulos.")

print("✓ Validaciones estructurales completadas.")


,Nulos,Porcentaje
IT_Futura_W1,0,0.0
IT_Futura_W2,0,0.0
IT_Futura_W3,0,0.0
IT_Futura_W4,0,0.0


✓ Las cuatro targets no contienen nulos.
✓ Validaciones estructurales completadas.


## 10. Exportación de la tabla de entrenamiento

La salida principal es:

**`Tabla_Entrenamiento_IT_Modelado.csv`**

Esta tabla constituye el **input del siguiente proceso de modelado** y contiene los predictores ya depurados y tratados junto con las cuatro variables objetivo.


In [19]:
OUTPUT_FILE = Path("Tabla_Entrenamiento_IT_Modelado.csv")

df_model.to_csv(
    OUTPUT_FILE,
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

summary = {
    "input": str(INPUT_FILE),
    "output": str(OUTPUT_FILE),
    "filas_input": int(len(df)),
    "columnas_input": int(len(df.columns)),
    "filas_output": int(len(df_model)),
    "columnas_output": int(len(df_model.columns)),
    "n_variables_candidatas": len(CANDIDATES),
    "n_variables_objetivo": len(TARGETS),
    "n_variables_output": len(df_model.columns),
    "imputacion_numericas": "Media",
    "imputacion_categoricas": "Moda",
    "edad_0_1_imputada_con_media": n_01,
    "edad_mayor_80_limitada_a_80": n_gt80,
    "edad_media_utilizada": edad_mean,
    "umbrales_reduccion_dimensionalidad": RARE_THRESHOLDS,
    "variables_agrupadas": list(RARE_THRESHOLDS.keys()),
    "encoding": "No aplicado en esta etapa; se realizará en el pipeline de modelado.",
    "scaling": "No aplicado en esta etapa; se decidirá según los algoritmos.",
}

with open(
    OUTPUT_DIR / "Resumen_Preparacion_IT.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(f"✓ Tabla de entrenamiento generada: {OUTPUT_FILE.resolve()}")
print(f"✓ Resumen generado: {(OUTPUT_DIR / 'Resumen_Preparacion_IT.json').resolve()}")


✓ Tabla de entrenamiento generada: C:\Users\Oscar\OneDrive\Documents\TFM\Tabla_Entrenamiento_IT_Modelado.csv
✓ Resumen generado: C:\Users\Oscar\OneDrive\Documents\TFM\Resultados_03_Preparacion_Datos_IT\Resumen_Preparacion_IT.json


## 11. Resultado final

La tabla `Tabla_Entrenamiento_IT_Modelado.csv` constituye el **input de la siguiente etapa de modelado**.

En esta etapa se han realizado exclusivamente transformaciones de preparación:

- retirada inicial de variables que no participarán en la predicción;
- tratamiento de valores faltantes;
- tratamiento de anomalías definidas para `Edad`;
- reducción de categorías poco frecuentes mediante umbrales diferenciados;
- análisis de las variables categóricas después del ajuste;
- reincorporación de las cuatro variables objetivo.

No se ha realizado todavía selección definitiva de predictores, One-Hot Encoding ni escalado. Estas decisiones se abordarán en la fase de modelado y validación.
